# Polaris availability test

Read-mostly health checks for a running Polaris. **This is the only suite that may be
pointed at a shared environment** — it does not call `require_not_prod`, by design.

It does write, but only inside the watchdog principal's own catalog
(`watchdog-catalog`), and cleans up after itself: T03/T04 create and drop a namespace
and a table there, and the final cell removes both. The catalog and principal are kept
for the next run.

Layer, matching the current notebooks: `polaris_test_utils` supplies the environment
and the token; `PolarisREST` makes the REST calls.

In [3]:
# -- src path bootstrap (shared modules live in /src) ---------------------------
import pathlib
import sys

_ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists())
_SRC = _ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import requests

from polaris_test_utils import *  # noqa: F401,F403
import polaris_test_utils as ptu

# init_env is NOT optional: BASE_MGMT / BASE_CAT / POLARIS_URL / REALM are module
# globals that start as None and are only assigned here. Skipping it is why every
# URL in the previous version of this notebook built as "None/watchdog-catalog/...".
init_env("local")   # "dev" / "prod" for the shared clusters -- this suite may target them

# Deliberately NO require_not_prod(): availability is the one suite allowed on PROD.

tr = TestResult()
print(f"env ready: {ptu.POLARIS_URL}  realm={ptu.REALM}")

🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.6.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
env ready: http://192.168.139.2:8181  realm=POLARIS


In [5]:
# Watchdog credentials -- never hardcoded, never committed.
#
# Set these in your shell before launching Jupyter:
#     export POLARIS_WATCHDOG_CLIENT_ID=...
#     export POLARIS_WATCHDOG_SECRET=...
#
# To mint them the first time, or after a 401 unauthorized_client (stale secret),
# run ONE of these by hand and export what it prints:
#     client_id, new_secret = ensure_watchdog_setup()       # fresh environment
#     client_id, new_secret = reset_watchdog_principal()    # on 401/403
import os

WATCHDOG_CLIENT_ID = os.environ.get("POLARIS_WATCHDOG_CLIENT_ID")
WATCHDOG_SECRET = os.environ.get("POLARIS_WATCHDOG_SECRET")

print(f"credentials loaded from environment for: {WATCHDOG_CLIENT_ID}")

assert WATCHDOG_CLIENT_ID and WATCHDOG_SECRET, (
    "POLARIS_WATCHDOG_CLIENT_ID / POLARIS_WATCHDOG_SECRET are not set -- see the "
    "comment above. They are read from the environment so no secret is ever "
    "written into this notebook."
)
print(f"credentials loaded from environment for: {WATCHDOG_CLIENT_ID}")

credentials loaded from environment for: None


AssertionError: POLARIS_WATCHDOG_CLIENT_ID / POLARIS_WATCHDOG_SECRET are not set -- see the comment above. They are read from the environment so no secret is ever written into this notebook.

In [ ]:
# T01: OAuth2 token acquisition (watchdog principal, default realm)
print("T01: OAuth2 token acquisition")
try:
    token = get_watchdog_token(WATCHDOG_CLIENT_ID, WATCHDOG_SECRET)
    tr.record("T01_auth", True, "token acquired")
except AssertionError as e:
    tr.record("T01_auth", False, str(e))
    print("Cannot continue without token")
    raise

In [ ]:
# REST clients. Two identities: the watchdog (what we are testing) and root
# (used only by T05, to name a catalog the watchdog must NOT be able to delete).
from polaris_rest import PolarisREST

wd = PolarisREST(ptu.POLARIS_URL, ptu.REALM, token=token)
adm = PolarisREST(ptu.POLARIS_URL, ptu.REALM, token=root_token())
print("clients ready: wd (watchdog), adm (root)")

In [ ]:
# T02: Watchdog catalog accessible (via catalog API)
# The management endpoint /api/management/v1/catalogs needs service_admin; the
# watchdog only has catalog-level access, so reach it through the catalog API.
print("\nT02: Watchdog catalog accessible")
r = wd.list_namespaces(WATCHDOG_CATALOG)
passed = r.status_code == 200
detail = f"HTTP {r.status_code}"
if passed:
    detail = f"catalog reachable, namespaces: {r.json().get('namespaces', [])}"
tr.record("T02_catalog_accessible", passed, detail)

In [ ]:
# T03: Namespace CRUD
print("\nT03: Namespace CRUD")

# Ensure clean state -- drop a leftover from a previous run
if wd.delete_namespace(WATCHDOG_CATALOG, WATCHDOG_NAMESPACE).status_code in (200, 204):
    print("  (cleaned up leftover namespace from previous run)")

r_create = wd.create_namespace(WATCHDOG_CATALOG, WATCHDOG_NAMESPACE)
passed_create = r_create.status_code in (200, 201)
tr.record(
    "T03a_namespace_create",
    passed_create,
    f"HTTP {r_create.status_code}"
    + ("" if passed_create else f" -- {r_create.json().get('error', {}).get('message', '')}"),
)

# Read back. Also the read-after-write check: PG-HA replica lag has produced a
# create that reports success and a get that 404s (see the replica-staleness probe).
r_get = wd.get_namespace(WATCHDOG_CATALOG, WATCHDOG_NAMESPACE)
passed_get = r_get.status_code == 200
tr.record("T03b_namespace_read", passed_get, f"HTTP {r_get.status_code}")

tr.record(
    "T03_namespace_crud",
    passed_create and passed_get,
    "create + read OK" if passed_create and passed_get else "FAILED",
)

In [ ]:
# T04: Table CRUD
print("\nT04: Table CRUD")

if wd.delete_table(WATCHDOG_CATALOG, WATCHDOG_NAMESPACE, WATCHDOG_TABLE).status_code in (200, 204):
    print("  (cleaned up leftover table from previous run)")

payload = {
    "name": WATCHDOG_TABLE,
    "location": (
        f"s3a://{ptu.BUCKET}/{WATCHDOG_CATALOG}/{WATCHDOG_NAMESPACE}/{WATCHDOG_TABLE}/"
    ),
    "schema": {
        "type": "struct",
        "fields": [
            {"id": 1, "name": "id", "type": "long", "required": True},
            {"id": 2, "name": "value", "type": "string", "required": False},
        ],
    },
}
r_create = wd.create_table(WATCHDOG_CATALOG, WATCHDOG_NAMESPACE, payload)
passed_create = r_create.status_code in (200, 201)
tr.record(
    "T04a_table_create",
    passed_create,
    f"HTTP {r_create.status_code}"
    + ("" if passed_create else f" -- {r_create.json().get('error', {}).get('message', '')}"),
)

r_get = wd.load_table(WATCHDOG_CATALOG, WATCHDOG_NAMESPACE, WATCHDOG_TABLE)
passed_get = r_get.status_code == 200
tr.record("T04b_table_read", passed_get, f"HTTP {r_get.status_code}")

tr.record(
    "T04_table_crud",
    passed_create and passed_get,
    "create + read OK" if passed_create and passed_get else "FAILED",
)

In [ ]:
# T05: RBAC enforcement (negative test)
# Name a real catalog with the root token, then try to DELETE it as the watchdog.
# Polaris must reach the RBAC check and answer 403. A 404 is ambiguous -- it means
# the catalog was not found, so RBAC was never evaluated and this proves nothing.
print("\nT05: RBAC enforcement -- watchdog must NOT delete real catalogs")

r_list = adm.list_catalogs()
targets = [c["name"] for c in r_list.json().get("catalogs", []) if c["name"] != WATCHDOG_CATALOG]

if not targets:
    tr.record("T05_rbac_enforcement", True, "skipped -- no other catalog exists to test against")
else:
    target = targets[0]
    r = wd.delete_catalog(target)
    passed = r.status_code == 403
    verdict = (
        "RBAC enforced"
        if passed
        else "RBAC BROKEN" if r.status_code in (200, 204) else "404=ambiguous, catalog may not exist"
    )
    tr.record("T05_rbac_enforcement", passed, f"HTTP {r.status_code} on DELETE {target} ({verdict})")

In [ ]:
# T06: Health correlation -- detect entity_version mismatch
print("\nT06: Health correlation")
r_health = requests.get(f"{ptu.POLARIS_URL}/q/health")
health_up = r_health.status_code == 200 and r_health.json().get("status") == "UP"
write_failed = not tr.results.get("T03_namespace_crud", {}).get("passed", True)

if write_failed and health_up:
    tr.record(
        "T06_health_correlation",
        False,
        "entity_version mismatch suspected -- health=UP but write failed",
    )
    print()
    print("ENTITY VERSION MISMATCH DETECTED")
    print("   Fix (ask the system engineer to run):")
    print("     kubectl rollout restart deployment/benchmarks-polaris -n datahub-hynix")
    print("   DO NOT restart PostgreSQL")
else:
    tr.record(
        "T06_health_correlation",
        True,
        f"health={'UP' if health_up else 'DOWN'} write={'OK' if not write_failed else 'FAIL'}",
    )

In [ ]:
# Cleanup -- only the watchdog's own table + namespace, never the catalog.
print("\nCleanup...")
wd.delete_table(WATCHDOG_CATALOG, WATCHDOG_NAMESPACE, WATCHDOG_TABLE)
wd.delete_namespace(WATCHDOG_CATALOG, WATCHDOG_NAMESPACE)
print("  table + namespace deleted (catalog + principal kept for next run)")

In [ ]:
# Summary
print()
all_passed = tr.summary()
print()
if all_passed:
    print("Polaris fully operational")
else:
    print("Issues detected -- check failed tests above")
    print()
    print("Common fixes (this notebook has no kubectl access -- report to the system engineer):")
    print("  T01 fail:            kubectl get pods -n datahub-hynix | grep polaris")
    print("  T03 fail + T06 warn: kubectl rollout restart deployment/benchmarks-polaris")
    print("  T05 fail:            CHECK RBAC IMMEDIATELY -- privileges are too broad")